# 在 Kaggle 上训练 Mamba-3 MiniGrid 记忆智能体 (顶级 P100 优化版)

此版本已针对 **NVIDIA P100** 和最新的 **Mamba-3** 架构进行了全方位适配。

推荐的 Kaggle 设置：
- **加速器 (Accelerator)**: 选择 **GPU P100**。
- **互联网 (Internet)**: 开启（On）。
- **Mamba 版本**: 2.3.2.post1+ (包含 Mamba-3 支持)。


In [ ]:
# ===== 用户配置 (P100 + Mamba-3 顶级版) =====
from pathlib import Path
import os

REPO_URL = "https://github.com/SShion0721/mamba-minigrid-memory.git"
REPO_BRANCH = "master"

WORKDIR = Path("/kaggle/working/mamba-minigrid-memory")
RUN_NAME = "kaggle_mamba3_p100_s11"

MODEL = "mamba"          
VARIANT = "mamba3"       # 使用最尖端的 Mamba-3 架构
ENV_ID = "MiniGrid-MemoryS11-v0"
TOTAL_STEPS = 5_000_000  
SEED = 42

# P100 + Mamba-3 优化参数
NUM_ENVS = 32            
NUM_STEPS = 128
CONTEXT_LEN = 64
CHUNK_LEN = 64
BATCH_CHUNKS = 16        
D_MODEL = 128
MAMBA_LAYERS = 2
D_STATE = 128            # Mamba-3 配合 SSD 架构，可以将状态维度推到 128 以上
D_CONV = 4
EXPAND = 2

SPATIAL_LAYERS = 2
SPATIAL_HEADS = 4

EVAL_INTERVAL = 100_000
EVAL_EPISODES = 50
SAVE_INTERVAL = 200_000
LOG_INTERVAL = 10

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")


In [ ]:
# ===== 硬件与环境检查 =====
import subprocess, sys
def run(cmd, check=True, cwd=None):
    print("$", cmd)
    return subprocess.run(cmd, shell=True, check=check, cwd=cwd)

run("nvidia-smi", check=False)
print("Python 版本:", sys.version)


In [ ]:
# ===== 安装最新依赖 (适配 Mamba-3) =====
run(f"{sys.executable} -m pip install -q --upgrade pip numpy") 
run(f"{sys.executable} -m pip install -q gymnasium minigrid tensorboard tqdm einops imageio[ffmpeg]")

if MODEL == "mamba":
    print("正在安装/编译最新版 mamba-ssm...")
    # 注意：2.3.2.post1 已支持 Mamba-3
    run(f"{sys.executable} -m pip install -q --no-cache-dir causal-conv1d mamba-ssm --no-build-isolation")

import torch, numpy as np
print("torch:", torch.__version__, "cuda:", torch.version.cuda)
print("numpy:", np.__version__)


In [ ]:
# ===== 获取项目代码 =====
import shutil
if WORKDIR.exists():
    print("清理旧代码...")
    shutil.rmtree(WORKDIR)
run(f"git clone --branch {REPO_BRANCH} --depth 1 {REPO_URL} {WORKDIR}")
assert (WORKDIR / "src" / "train_mamba_ppo.py").exists()


In [ ]:
# ===== 启动 Mamba-3 训练 =====
cmd = f"""
{sys.executable} src/train_mamba_ppo.py
  --model {MODEL}
  --mamba-variant {VARIANT}
  --env-id {ENV_ID}
  --seed {SEED}
  --total-steps {TOTAL_STEPS}
  --num-envs {NUM_ENVS}
  --num-steps {NUM_STEPS}
  --context-len {CONTEXT_LEN}
  --chunk-len {CHUNK_LEN}
  --batch-chunks {BATCH_CHUNKS}
  --d-model {D_MODEL}
  --spatial-layers {SPATIAL_LAYERS}
  --spatial-heads {SPATIAL_HEADS}
  --mamba-layers {MAMBA_LAYERS}
  --d-state {D_STATE}
  --d-conv {D_CONV}
  --expand {EXPAND}
  --lr 2.5e-4
  --eval-interval {EVAL_INTERVAL}
  --save-interval {SAVE_INTERVAL}
  --run-name {RUN_NAME}
""".replace("\n", " ")
run(cmd, cwd=WORKDIR)


### 优化说明
1. **Mamba-3 适配**: 已经在代码中加入了 `mamba3` 的解析逻辑。
2. **极限状态维度**: 对于 Mamba-3，我们将 `D_STATE` 设为 **128**，这能提供前所未有的记忆容量，帮助智能体攻克 S13Random/S17Random 等极难关卡。
